# B4c — Frozen candidates: reproduction test and second held-out evaluation

Second, pre-declared held-out comparison of the five improvement-phase candidates, H1b_v1 and B0.

- **Part A** (done 14 Oct 2026, CPU): the frozen module `src/pipeline/candidates.py` reproduced the train_dev400 per-image counts of B3PN and B3AC exactly (9/9 comparisons, 400/400); frozen config JSONs written.
- **Part B** (CPU, resumable): candidate counts on held-out (1,186) and dev100 (100); the pre-declared rule is written to Drive first.
- **Part C** (CPU): metrics, paired bootstrap, pre-declared selection on held-out; full val reported for comparison.

Run order for Parts B–C in a fresh session: Steps 0.1, 0.2, then B.0 → C.5 (Part A need not be re-run).

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, repo, frozen module
# ============================================================
import importlib, json, re, subprocess, sys, time, shutil
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
OUT_DIR = RESULTS_ROOT / "B4c_HeldOut_Candidates"
(OUT_DIR / "configs").mkdir(parents=True, exist_ok=True)
SG_CFG_PATH = RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json"

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.candidates as C
for m in (io_cache, sg, C):
    importlib.reload(m)
SG_CFG = sg.SGConfig.from_frozen_json(SG_CFG_PATH)
assert SG_CFG.variant == "v1.1"

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv").sort_values("dataset_index").reset_index(drop=True)
assert len(td_df) == 400
TD_PATHS = C.CandidatePaths.standard(TD_ROOT)
print("Repo:", REPO_COMMIT[:10], "| module:", C.VERSION, "| candidates:", C.CANDIDATE_NAMES)

Repo: 84da2741b2 | module: candidates_v1 | candidates: ['GPV_R4+P[MixA]', 'GPV_K-n+P[MixA]', 'GPV_K-n+P[LM0]', 'H1b+P[MixA]', 'H1b-K3n+P[LM0]']


In [ ]:
# ============================================================
# Step A.1 — Five candidates on train_dev400 (resumable per image)
# ============================================================
ROWS = OUT_DIR / "A_train_rows"
ROWS.mkdir(parents=True, exist_ok=True)
todo = [r for r in td_df.itertuples(index=False) if not (ROWS / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for r in tqdm(todo, desc="Candidates (train pool)"):
    inp = C.load_image_inputs(TD_PATHS, r.dataset_index, r.image_id)
    out = C.compute_candidates(inp, SG_CFG)
    with open(ROWS / f"{int(r.dataset_index):05d}.json", "w") as f:
        json.dump({"dataset_index": int(r.dataset_index), **{k: (None if isinstance(v, float) and np.isnan(v) else v)
                                                           for k, v in out.items()}}, f)
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Candidates (train pool):   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 29.3 min


In [ ]:
# ============================================================
# Step A.2 — Reproduction test (exact) vs B3PN and B3AC
#            (rows copied to local disk first: many small Drive reads can stall)
# ============================================================
LOCAL = Path("/content/A_rows_local")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(ROWS, LOCAL)
mod = pd.DataFrame([json.load(open(p)) for p in sorted(LOCAL.glob("*.json"))])
assert len(mod) == 400, len(mod)

refs = {"B3PN": pd.read_csv(RESULTS_ROOT / "B3PN_Part_and_Normals_Merge" / "B3PN_per_image.csv"),
        "B3AC": pd.read_csv(RESULTS_ROOT / "B3AC_Area_Cap_and_P_on_S" / "B3AC_per_image.csv")}
rep, REPRO_PASS = [], True
for src, ref in refs.items():
    for c in C.CANDIDATE_NAMES:
        if c not in ref.columns:
            continue
        j = mod[["dataset_index", c]].merge(ref[["dataset_index", c]], on="dataset_index",
                                            suffixes=("_mod", "_ref"), validate="one_to_one")
        eq = (j[f"{c}_mod"].astype(float) == j[f"{c}_ref"].astype(float))
        rep.append({"reference": src, "candidate": c, "equal": int(eq.sum()), "of": len(j),
                    "first_diffs": j.loc[~eq, "dataset_index"].head(5).tolist()})
        REPRO_PASS &= bool(eq.all()) and len(j) == 400
rep = pd.DataFrame(rep)
display(rep)
covered = set(rep["candidate"])
assert covered == set(C.CANDIDATE_NAMES), f"not covered: {set(C.CANDIDATE_NAMES) - covered}"
print("REPRODUCTION TEST:", "PASS" if REPRO_PASS else "FAIL — stop, do not run held-out")

,reference,candidate,equal,of,first_diffs
0,B3PN,GPV_R4+P[MixA],400,400,[]
1,B3PN,GPV_K-n+P[MixA],400,400,[]
2,B3PN,GPV_K-n+P[LM0],400,400,[]
3,B3PN,H1b-K3n+P[LM0],400,400,[]
4,B3AC,GPV_R4+P[MixA],400,400,[]
5,B3AC,GPV_K-n+P[MixA],400,400,[]
6,B3AC,GPV_K-n+P[LM0],400,400,[]
7,B3AC,H1b+P[MixA],400,400,[]
8,B3AC,H1b-K3n+P[LM0],400,400,[]


REPRODUCTION TEST: PASS


In [ ]:
# ============================================================
# Step A.3 — Save: per-image counts, frozen configs, metadata
# ============================================================
assert REPRO_PASS
mod.to_csv(OUT_DIR / "B4c_A_train_pool_module_counts.csv", index=False)
rep.to_csv(OUT_DIR / "B4c_A_reproduction.csv", index=False)
for c in C.CANDIDATE_NAMES:
    fn = re.sub(r"[^A-Za-z0-9]+", "_", c).strip("_")          # e.g. GPV_K_n_P_MixA
    with open(OUT_DIR / "configs" / f"{fn}_frozen_config.json", "w") as f:
        json.dump(C.frozen_config(c, SG_CFG_PATH), f, indent=2)
with open(OUT_DIR / "B4c_A_metadata.json", "w") as f:
    json.dump({"notebook": "B4c_HeldOut_Candidates (Part A)", "module": C.VERSION,
               "repo_commit": REPO_COMMIT, "reproduction_pass": REPRO_PASS,
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR); print(sorted(p.name for p in (OUT_DIR / "configs").glob("*.json")))

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B4c_HeldOut_Candidates
['GPV_K_n_P_LM0_frozen_config.json', 'GPV_K_n_P_MixA_frozen_config.json', 'GPV_R4_P_MixA_frozen_config.json', 'H1b_K3n_P_LM0_frozen_config.json', 'H1b_P_MixA_frozen_config.json']


# Part B — Held-out and dev100 counts for the five candidates

**Pre-declared rule (agreed 14 Oct 2026, recorded in Step B.0 before any validation count is computed).**

- **Decision set:** held-out 1,186. Full val 1,286 (held-out + dev100) is reported for comparison only. The FSC-147 test set is not touched.
- **Systems:** eligible = the five candidates + H1b_v1 (no special incumbent status). B0 = reference only. B0 and H1b_v1 counts are taken unchanged from B4 (`B4_C_heldout_per_image.csv`, `B4_C_fullval_per_image.csv`).
- **Metrics:** MAE (primary), RMSE, trimmed MAE (own 1%), NAE, mean signed error, per-stratum MAE. Paired bootstrap 10,000, seed 0, per-stratum seeds 1000 + j (as B4).
- **Stage 1 — performance:** leader = lowest MAE; tied set = systems whose MAE-difference CI vs the leader includes 0; drop if RMSE significantly worse than the best RMSE in the tied set.
- **Stage 2 — stability:** (i) drop if trimmed MAE significantly worse than the best in the current set; (ii) drop if significantly worse than **H1b_v1** in **≥ 2 of 6** strata; (iii) drop if overall MAE significantly worse than **H1b_v1**. H1b_v1 is exempt from (ii)–(iii). If Stage 2 removes every system, keep those with the fewest Stage-2 failures (B4 tie handling).
- **Stage 3 — adaptability then priors then cost:** GPV_K-n+P[LM0] → GPV_K-n+P[MixA] → GPV_R4+P[MixA] → H1b+P[MixA] → H1b-K3n+P[LM0] → H1b_v1 (keys: GPV family before H1b; both priors before semantic-only; fewer SAM2 passes).
- **Outputs:** PRIMARY = first survivor in the Stage-3 order; FALLBACK = next survivor, else the eliminated eligible system with the lowest MAE (as B4).
- **Reported regardless:** all seven systems; B0 per stratum and each system's gap to B0 on S1–S2; gap to H1b_v1 on S5–S6; secondary analysis GPV_K-n+P[MixA] − GPV_R4+P[MixA] (geometric-prior contribution); full-val leader vs held-out leader. No re-runs with changed parameters.
- This is the **second, pre-declared** use of the held-out set (the first was B4).

In [3]:
# ============================================================
# Step B.0 — Record the pre-declared rule (before any validation count)
# ============================================================
assert REPRO_PASS if "REPRO_PASS" in globals() else json.load(open(OUT_DIR / "B4c_A_metadata.json"))["reproduction_pass"]

REFERENCES = ["B0", "H1b_v1"]
ELIGIBLE = C.CANDIDATE_NAMES + ["H1b_v1"]
SYSTEMS_ALL = ["B0"] + ELIGIBLE
STAGE3_ORDER = ["GPV_K-n+P[LM0]", "GPV_K-n+P[MixA]", "GPV_R4+P[MixA]",
                "H1b+P[MixA]", "H1b-K3n+P[LM0]", "H1b_v1"]
INCUMBENT = "H1b_v1"
STRATUM_FAIL_MIN = 2
N_BOOT = 10_000
assert sorted(STAGE3_ORDER) == sorted(ELIGIBLE)

RULE = {
    "decision_set": "FSC-147 val held-out (1,186)",
    "comparison_set": "full val (1,286) = held-out + dev100; reported only",
    "eligible": ELIGIBLE, "reference_only": ["B0"],
    "reference_counts": "B0, H1b_v1 from B4 partC CSVs, unchanged",
    "bootstrap": {"n": N_BOOT, "seed": 0, "stratum_seeds": "1000 + j"},
    "stage1": "leader = lowest MAE; tied = MAE-diff CI vs leader includes 0; "
              "drop if RMSE significantly worse than best RMSE in tied set",
    "stage2": ["drop if trimmed MAE significantly worse than best in current set",
               f"drop if significantly worse than {INCUMBENT} in >= {STRATUM_FAIL_MIN} of 6 strata",
               f"drop if overall MAE significantly worse than {INCUMBENT}",
               f"{INCUMBENT} exempt from the two incumbent tests",
               "if all removed: keep those with fewest stage-2 failures"],
    "stage3_order": STAGE3_ORDER,
    "stage3_keys": ["GPV family before H1b family (multi-label adaptation)",
                    "both semantic and geometric priors before semantic-only",
                    "fewer SAM2 passes (LM0 before MixA)"],
    "fallback": "next survivor in stage-3 order, else eliminated eligible system with lowest MAE",
    "secondary_analysis": "GPV_K-n+P[MixA] - GPV_R4+P[MixA], held-out and full val",
    "second_use_of_heldout": True,
}
RULE_PATH = OUT_DIR / "B4c_predeclared_rule.json"
if RULE_PATH.exists():
    saved = json.load(open(RULE_PATH))
    assert {k: v for k, v in saved.items() if k != "recorded_utc"} == RULE, "Rule changed after recording — stop."
    print("Rule already recorded at", saved["recorded_utc"], "— unchanged")
else:
    with open(RULE_PATH, "w") as f:
        json.dump({**RULE, "recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, f, indent=2)
    print("Rule recorded:", RULE_PATH)

Rule already recorded at 2026-10-14T08:53:04+00:00 — unchanged


In [4]:
# ============================================================
# Step B.1 — Subsets, cache paths, B4 reference counts
# ============================================================
HO_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_heldout_1186"
DEV_ROOT = PROJECT_ROOT / "cache" / "FSC147_val_dev100_candidates"
FRESH_ROOT = RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "section3_end_to_end" / "fresh_dev100_cache"
B4_C = RESULTS_ROOT / "B4_HeldOut_FSC147_Validation" / "partC_evaluation"

HO_PATHS = C.CandidatePaths.standard(HO_ROOT)
DEV_PATHS = C.CandidatePaths(root=DEV_ROOT, san_dir=FRESH_ROOT / "san",           # B3R Option A
                             a3c_dir=FRESH_ROOT / "a3c_masks", depth_dir=FRESH_ROOT / "depth")

META = ["dataset_index", "image_id", "category", "gt_count", "density_stratum"]
ref_ho = pd.read_csv(B4_C / "B4_C_heldout_per_image.csv")[META + REFERENCES]
ref_full = pd.read_csv(B4_C / "B4_C_fullval_per_image.csv")
ref_dev = ref_full[ref_full["subset"] == "dev100"][META + REFERENCES].reset_index(drop=True)
assert len(ref_ho) == 1186 and len(ref_dev) == 100 and len(ref_full) == 1286

ho_list = pd.read_csv(HO_ROOT / "FSC147_val_heldout_1186.csv")
assert set(ho_list["dataset_index"]) == set(ref_ho["dataset_index"])
chk = ref_full[ref_full["subset"] == "heldout"].merge(ref_ho, on="dataset_index", suffixes=("_f", "_h"))
assert all(np.allclose(chk[f"{s}_f"], chk[f"{s}_h"]) for s in REFERENCES), "B4 held-out vs full-val rows disagree"

STRATA = sorted(ref_ho["density_stratum"].unique())
assert len(STRATA) == 6, STRATA
print("Strata:", STRATA)
print(ref_ho.groupby("density_stratum").size().to_dict(), "| dev100:", ref_dev.groupby("density_stratum").size().to_dict())

Strata: ['S1_0_20', 'S2_20_40', 'S3_40_60', 'S4_60_80', 'S5_80_95', 'S6_95_100']
{'S1_0_20': 238, 'S2_20_40': 237, 'S3_40_60': 237, 'S4_60_80': 237, 'S5_80_95': 182, 'S6_95_100': 55} | dev100: {'S1_0_20': 20, 'S2_20_40': 20, 'S3_40_60': 20, 'S4_60_80': 20, 'S5_80_95': 10, 'S6_95_100': 10}


In [5]:
# ============================================================
# Step B.2 — Candidates on held-out (1,186) and dev100 (100), resumable
# ============================================================
def run_subset(name, image_df, paths):
    rows_dir = OUT_DIR / f"B_{name}_rows"
    rows_dir.mkdir(parents=True, exist_ok=True)
    todo = [r for r in image_df.itertuples(index=False)
            if not (rows_dir / f"{int(r.dataset_index):05d}.json").exists()]
    print(f"[{name}] done: {len(image_df) - len(todo)} | to process: {len(todo)}")
    t0 = time.time()
    for r in tqdm(todo, desc=f"Candidates ({name})"):
        out = C.compute_candidates(C.load_image_inputs(paths, r.dataset_index, r.image_id), SG_CFG)
        with open(rows_dir / f"{int(r.dataset_index):05d}.json", "w") as f:
            json.dump({"dataset_index": int(r.dataset_index),
                       **{k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in out.items()}}, f)
    print(f"[{name}] elapsed: {(time.time() - t0) / 60:.1f} min")
    local = Path(f"/content/B_{name}_rows_local")       # many small Drive reads can stall
    if local.exists():
        shutil.rmtree(local)
    shutil.copytree(rows_dir, local)
    df = pd.DataFrame([json.load(open(p)) for p in sorted(local.glob("*.json"))])
    assert len(df) == len(image_df), (name, len(df))
    return df

cand_ho = run_subset("heldout", ref_ho[["dataset_index", "image_id"]], HO_PATHS)
cand_dev = run_subset("dev100", ref_dev[["dataset_index", "image_id"]], DEV_PATHS)

[heldout] done: 1186 | to process: 0


Candidates (heldout): 0it [00:00, ?it/s]

[heldout] elapsed: 0.0 min
[dev100] done: 100 | to process: 0


Candidates (dev100): 0it [00:00, ?it/s]

[dev100] elapsed: 0.0 min


In [6]:
# ============================================================
# Step B.3 — Merge candidates with B4 references
# ============================================================
ho = ref_ho.merge(cand_ho, on="dataset_index", validate="one_to_one")
dv = ref_dev.merge(cand_dev, on="dataset_index", validate="one_to_one")
full = pd.concat([ho, dv], ignore_index=True)
assert len(ho) == 1186 and len(dv) == 100 and len(full) == 1286
assert not full[SYSTEMS_ALL].isna().any().any()
print("Merged: held-out", len(ho), "| dev100", len(dv), "| full val", len(full))
print("Empty R4 sets (held-out):", {b: int((ho[f"n_r4[{b}]"] == 0).sum()) for b in C.BASES})

Merged: held-out 1186 | dev100 100 | full val 1286
Empty R4 sets (held-out): {'MixA': 20, 'LM0': 20}


# Part C — Metrics, bootstrap, pre-declared selection

In [7]:
# ============================================================
# Step C.1 — Metrics: held-out (decision), full val and dev100 (comparison)
# ============================================================
def role(s):
    return "reference" if s == "B0" else "incumbent" if s == INCUMBENT else "candidate"

def report(df, label):
    gt = df["gt_count"].to_numpy(float); st = df["density_stratum"].to_numpy()
    k = max(1, int(round(0.01 * len(df))))
    rows = []
    for s in SYSTEMS_ALL:
        err = df[s].to_numpy(float) - gt; ae = np.abs(err)
        row = {"set": label, "system": s, "role": role(s), "MAE": ae.mean(),
               "RMSE": np.sqrt((err ** 2).mean()), "trimmed_MAE": np.sort(ae)[:len(ae) - k].mean(),
               "NAE": (ae / gt).mean(), "mean_signed_error": err.mean()}
        for sname in STRATA:
            row[f"MAE_{sname[:2]}"] = ae[st == sname].mean() if (st == sname).any() else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

rep_ho, rep_full, rep_dev = report(ho, "held-out (1,186)"), report(full, "full val (1,286)"), report(dv, "dev100 (100)")
SCOLS = [f"MAE_{s[:2]}" for s in STRATA]
COLS_R = ["system", "role", "MAE", "RMSE", "trimmed_MAE", "NAE", "mean_signed_error"] + SCOLS
print("HELD-OUT (decision set)"); display(rep_ho[COLS_R].round(2))
print("\nFULL VALIDATION (comparison only; includes dev100)"); display(rep_full[COLS_R].round(2))

M = rep_ho.set_index("system")
gaps = pd.DataFrame({
    **{f"{s}: − B0": M[s] - M.loc["B0", s] for s in SCOLS[:2]},
    **{f"{s}: − H1b_v1": M[s] - M.loc[INCUMBENT, s] for s in SCOLS[4:]},
}).loc[SYSTEMS_ALL]
print("\nHeld-out stratum gaps (T1-style vs B0 on S1–S2; T2-style vs H1b_v1 on S5–S6; negative = better):")
display(gaps.round(2))

shift = (rep_ho.set_index("system")["MAE"] - rep_dev.set_index("system")["MAE"]).loc[SYSTEMS_ALL]
print("\nHeld-out MAE minus dev100 MAE (negative = lower error on held-out):")
display(shift.round(2).to_frame("held-out − dev100"))

HELD-OUT (decision set)


,system,role,MAE,RMSE,trimmed_MAE,NAE,mean_signed_error,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,reference,38.35,102.23,31.04,0.53,-34.82,6.03,8.02,11.78,25.95,74.67,356.58
1,GPV_R4+P[MixA],candidate,27.07,88.33,20.23,0.42,-15.13,6.74,7.00,8.86,14.76,41.10,286.53
2,GPV_K-n+P[MixA],candidate,29.05,87.40,22.46,0.59,-11.59,13.23,10.15,9.49,17.01,39.76,279.62
3,GPV_K-n+P[LM0],candidate,28.43,85.73,22.35,0.38,-19.45,4.50,7.13,7.71,16.24,48.53,299.07
4,H1b+P[MixA],candidate,27.56,86.87,20.85,0.48,-13.10,9.05,7.46,9.12,16.51,40.85,277.38
5,H1b-K3n+P[LM0],candidate,29.14,94.42,21.99,0.35,-20.24,4.03,5.56,7.61,16.07,49.55,320.89
6,H1b_v1,incumbent,29.99,90.60,23.07,0.52,-16.12,9.58,8.82,10.33,14.61,44.90,311.18



FULL VALIDATION (comparison only; includes dev100)


,system,role,MAE,RMSE,trimmed_MAE,NAE,mean_signed_error,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0,reference,41.64,123.76,32.35,0.53,-38.13,5.96,7.90,12.03,26.60,74.02,397.55
1,GPV_R4+P[MixA],candidate,30.01,110.55,21.33,0.41,-18.36,6.57,7.01,8.89,14.51,40.40,328.20
2,GPV_K-n+P[MixA],candidate,32.39,110.90,23.91,0.62,-14.40,13.13,12.13,9.49,16.67,39.12,321.78
3,GPV_K-n+P[LM0],candidate,31.31,108.70,23.26,0.38,-22.56,4.59,6.86,7.78,16.04,47.76,338.92
4,H1b+P[MixA],candidate,29.79,106.34,21.50,0.47,-15.74,8.70,7.47,9.05,16.02,40.14,307.72
5,H1b-K3n+P[LM0],candidate,31.84,114.60,22.90,0.35,-23.24,4.16,5.44,7.67,15.66,48.62,356.15
6,H1b_v1,incumbent,32.37,110.54,23.79,0.51,-18.88,9.17,8.75,10.27,14.65,44.28,340.03



Held-out stratum gaps (T1-style vs B0 on S1–S2; T2-style vs H1b_v1 on S5–S6; negative = better):


,MAE_S1: − B0,MAE_S2: − B0,MAE_S5: − H1b_v1,MAE_S6: − H1b_v1
system,,,,
B0,0.00,0.00,29.77,45.40
GPV_R4+P[MixA],0.72,-1.02,-3.79,-24.65
GPV_K-n+P[MixA],7.21,2.14,-5.14,-31.56
GPV_K-n+P[LM0],-1.53,-0.89,3.63,-12.11
H1b+P[MixA],3.02,-0.56,-4.05,-33.80
H1b-K3n+P[LM0],-2.00,-2.46,4.65,9.71
H1b_v1,3.55,0.80,0.00,0.00



Held-out MAE minus dev100 MAE (negative = lower error on held-out):


,held-out − dev100
system,
B0,-42.34
GPV_R4+P[MixA],-37.88
GPV_K-n+P[MixA],-43.02
GPV_K-n+P[LM0],-37.06
H1b+P[MixA],-28.75
H1b-K3n+P[LM0],-34.82
H1b_v1,-30.58


In [8]:
# ============================================================
# Step C.2 — Paired bootstrap (10,000 resamples, seed 0; strata seeds 1000 + j)
# ============================================================
def boot_stats(df):
    gt = df["gt_count"].to_numpy(float); st = df["density_stratum"].to_numpy()
    n = len(df); k = max(1, int(round(0.01 * n)))
    idx = np.random.default_rng(0).integers(0, n, size=(N_BOOT, n))
    strat_idx = {}
    for j, s in enumerate(STRATA):
        pos = np.flatnonzero(st == s)
        strat_idx[s] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(N_BOOT, len(pos)))]
    B = {}
    for s in SYSTEMS_ALL:
        ae = np.abs(df[s].to_numpy(float) - gt); sample = ae[idx]
        B[s] = {"MAE": sample.mean(axis=1), "RMSE": np.sqrt((sample ** 2).mean(axis=1)),
                "trimmed_MAE": np.sort(sample, axis=1)[:, :n - k].mean(axis=1),
                **{f"MAE_{x[:2]}": ae[strat_idx[x]].mean(axis=1) for x in STRATA}}
    return B

def ci(B, a, b, metric):
    d = B[a][metric] - B[b][metric]
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

t0 = time.time()
BOOT_HO, BOOT_FULL = boot_stats(ho), boot_stats(full)
print(f"Bootstrap done in {time.time() - t0:.0f} s")

def pair_table(B, rep, systems, metrics):
    R = rep.set_index("system"); rows = []
    for i, a in enumerate(systems):
        for b in systems[i + 1:]:
            for m in metrics:
                lo, hi = ci(B, a, b, m)
                rows.append({"A": a, "B": b, "metric": m, "A−B": R.loc[a, m] - R.loc[b, m],
                             "CI_low": lo, "CI_high": hi, "significant": not (lo <= 0 <= hi)})
    return pd.DataFrame(rows)

pairs_ho = pair_table(BOOT_HO, rep_ho, SYSTEMS_ALL, ["MAE", "RMSE", "trimmed_MAE"])
print("Held-out paired MAE differences (A − B; negative = A better):")
display(pairs_ho[pairs_ho["metric"] == "MAE"].round(2))

def vs_table(B, rep, ref, metrics):
    R = rep.set_index("system"); rows = []
    for s in SYSTEMS_ALL:
        if s == ref: continue
        row = {"system": s}
        for m in metrics:
            lo, hi = ci(B, s, ref, m)
            row[m] = f"{R.loc[s, m] - R.loc[ref, m]:+.2f} [{lo:+.2f}, {hi:+.2f}]" + (" *" if not lo <= 0 <= hi else "")
        rows.append(row)
    return pd.DataFrame(rows).set_index("system")

vs_h1b = vs_table(BOOT_HO, rep_ho, INCUMBENT, ["MAE"] + SCOLS)
vs_b0 = vs_table(BOOT_HO, rep_ho, "B0", ["MAE"] + SCOLS)
print("\nHeld-out: system − H1b_v1, 95% CI (* = significant)"); display(vs_h1b)
print("\nHeld-out: system − B0, 95% CI (reference; * = significant)"); display(vs_b0)

Bootstrap done in 5 s
Held-out paired MAE differences (A − B; negative = A better):


,A,B,metric,A−B,CI_low,CI_high,significant
0,B0,GPV_R4+P[MixA],MAE,11.28,9.54,13.15,True
3,B0,GPV_K-n+P[MixA],MAE,9.30,7.17,11.47,True
6,B0,GPV_K-n+P[LM0],MAE,9.92,7.83,12.53,True
9,B0,H1b+P[MixA],MAE,10.79,8.92,12.79,True
12,B0,H1b-K3n+P[LM0],MAE,9.21,7.73,10.69,True
15,B0,H1b_v1,MAE,8.36,7.08,9.72,True
18,GPV_R4+P[MixA],GPV_K-n+P[MixA],MAE,-1.98,-2.95,-1.03,True
21,GPV_R4+P[MixA],GPV_K-n+P[LM0],MAE,-1.36,-2.93,0.55,False
24,GPV_R4+P[MixA],H1b+P[MixA],MAE,-0.49,-0.91,-0.08,True
27,GPV_R4+P[MixA],H1b-K3n+P[LM0],MAE,-2.07,-3.31,-0.96,True



Held-out: system − H1b_v1, 95% CI (* = significant)


,MAE,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
system,,,,,,,
B0,"+8.36 [+7.08, +9.72] *","-3.55 [-5.00, -2.24] *","-0.80 [-2.35, +0.63]","+1.45 [-0.16, +3.03]","+11.35 [+8.88, +13.83] *","+29.77 [+26.09, +33.47] *","+45.40 [+34.20, +57.58] *"
GPV_R4+P[MixA],"-2.92 [-3.98, -1.91] *","-2.84 [-4.17, -1.62] *","-1.82 [-2.67, -1.02] *","-1.47 [-2.49, -0.47] *","+0.15 [-1.33, +1.70]","-3.79 [-7.27, -0.07] *","-24.65 [-39.58, -10.29] *"
GPV_K-n+P[MixA],"-0.94 [-2.38, +0.51]","+3.65 [+0.18, +7.46] *","+1.33 [-0.96, +4.14]","-0.85 [-2.16, +0.60]","+2.41 [+0.63, +4.32] *","-5.14 [-8.80, -1.27] *","-31.56 [-47.71, -16.45] *"
GPV_K-n+P[LM0],"-1.56 [-3.74, +0.07]","-5.08 [-6.40, -3.89] *","-1.69 [-3.60, +0.83]","-2.62 [-3.64, -1.59] *","+1.63 [+0.00, +3.33] *","+3.63 [+0.40, +7.09] *","-12.11 [-56.67, +17.22]"
H1b+P[MixA],"-2.43 [-3.57, -1.36] *","-0.53 [-1.95, +0.90]","-1.36 [-2.20, -0.57] *","-1.22 [-2.22, -0.19] *","+1.90 [+0.22, +3.71] *","-4.05 [-7.47, -0.38] *","-33.80 [-48.95, -19.45] *"
H1b-K3n+P[LM0],"-0.85 [-1.81, +0.19]","-5.55 [-6.74, -4.50] *","-3.26 [-4.41, -1.81] *","-2.72 [-3.73, -1.72] *","+1.46 [-0.13, +3.15]","+4.65 [+1.79, +7.76] *","+9.71 [-3.75, +26.02]"



Held-out: system − B0, 95% CI (reference; * = significant)


,MAE,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
system,,,,,,,
GPV_R4+P[MixA],"-11.28 [-13.15, -9.54] *","+0.72 [-0.36, +1.84]","-1.02 [-2.40, +0.52]","-2.92 [-4.53, -1.30] *","-11.19 [-14.20, -8.18] *","-33.57 [-38.94, -28.23] *","-70.05 [-91.64, -49.80] *"
GPV_K-n+P[MixA],"-9.30 [-11.47, -7.17] *","+7.21 [+3.69, +11.10] *","+2.14 [-0.65, +5.46]","-2.30 [-4.12, -0.36] *","-8.94 [-11.96, -5.90] *","-34.91 [-40.44, -29.43] *","-76.96 [-99.84, -55.73] *"
GPV_K-n+P[LM0],"-9.92 [-12.53, -7.83] *","-1.53 [-2.62, -0.34] *","-0.89 [-3.21, +2.08]","-4.07 [-5.60, -2.57] *","-9.71 [-12.21, -7.21] *","-26.14 [-30.95, -21.46] *","-57.51 [-106.64, -24.34] *"
H1b+P[MixA],"-10.79 [-12.79, -8.92] *","+3.02 [+1.34, +4.86] *","-0.56 [-2.02, +1.04]","-2.67 [-4.30, -1.05] *","-9.45 [-12.41, -6.42] *","-33.82 [-39.23, -28.40] *","-79.20 [-101.64, -58.33] *"
H1b-K3n+P[LM0],"-9.21 [-10.69, -7.73] *","-2.00 [-2.88, -1.11] *","-2.46 [-4.02, -0.31] *","-4.17 [-5.62, -2.75] *","-9.89 [-12.42, -7.32] *","-25.12 [-29.91, -20.45] *","-35.69 [-54.75, -15.29] *"
H1b_v1,"-8.36 [-9.72, -7.08] *","+3.55 [+2.24, +5.00] *","+0.80 [-0.63, +2.35]","-1.45 [-3.03, +0.16]","-11.35 [-13.83, -8.88] *","-29.77 [-33.47, -26.09] *","-45.40 [-57.58, -34.20] *"


In [9]:
# ============================================================
# Step C.3 — Pre-declared selection procedure (held-out)
# ============================================================
M = rep_ho.set_index("system")
log = []
worse = lambda a, b, m: ci(BOOT_HO, a, b, m)[0] > 0          # a significantly worse than b

# Stage 1 — performance
leader = min(ELIGIBLE, key=lambda s: M.loc[s, "MAE"])
tied = [leader] + [c for c in ELIGIBLE if c != leader
                   and ci(BOOT_HO, c, leader, "MAE")[0] <= 0 <= ci(BOOT_HO, c, leader, "MAE")[1]]
log.append(f"Stage 1 leader (lowest MAE): {leader} ({M.loc[leader, 'MAE']:.2f})")
log.append(f"Stage 1 tied set (MAE CI vs leader includes 0): {tied}")
best_rmse = min(tied, key=lambda s: M.loc[s, "RMSE"])
stage1 = [c for c in tied if c == best_rmse or not worse(c, best_rmse, "RMSE")]
log.append(f"Stage 1 after RMSE filter (best RMSE {best_rmse}): {stage1}")

# Stage 2 — stability
best_trim = min(stage1, key=lambda s: M.loc[s, "trimmed_MAE"])
fails = {c: [] for c in stage1}
for c in stage1:
    if c != best_trim and worse(c, best_trim, "trimmed_MAE"):
        fails[c].append(f"trimmed MAE worse than {best_trim}")
    if c != INCUMBENT:
        sf = [s[:2] for s in STRATA if worse(c, INCUMBENT, f"MAE_{s[:2]}")]
        if len(sf) >= STRATUM_FAIL_MIN:
            fails[c].append(f"worse than {INCUMBENT} in {sf}")
        elif sf:
            log.append(f"  note: {c} worse than {INCUMBENT} in {sf} only (< {STRATUM_FAIL_MIN}, kept)")
        if worse(c, INCUMBENT, "MAE"):
            fails[c].append(f"overall MAE worse than {INCUMBENT}")
stage2 = [c for c in stage1 if not fails[c]]
log.append("Stage 2 failures: " + "; ".join(f"{c}: {fails[c] or 'none'}" for c in stage1))
if not stage2:
    fewest = min(len(v) for v in fails.values())
    stage2 = [c for c in stage1 if len(fails[c]) == fewest]
    log.append("Stage 2 removed every system — kept those with the fewest failures")
log.append(f"Stage 2 remaining: {stage2}")

# Stage 3 — adaptability, priors, cost
survivors = [c for c in STAGE3_ORDER if c in stage2]
primary = survivors[0]
fallback = survivors[1] if len(survivors) > 1 else \
    min([c for c in ELIGIBLE if c != primary], key=lambda s: M.loc[s, "MAE"])
log.append(f"Stage 3 order {STAGE3_ORDER}")
log.append(f"PRIMARY = {primary}; FALLBACK = {fallback}"
           + ("" if len(survivors) > 1 else " (only one survivor: lowest-MAE other eligible)"))

clear = [c for c in ELIGIBLE if all(ci(BOOT_HO, c, o, "MAE")[1] < 0 for o in ELIGIBLE if o != c)]
log.append(f"Clear winner (MAE better than every other eligible, CI excludes 0): {clear or 'none'}")
lo, hi = ci(BOOT_HO, primary, INCUMBENT, "MAE") if primary != INCUMBENT else (0.0, 0.0)
claim = ("significantly better than H1b_v1" if hi < 0 else
         "not significantly different from H1b_v1" if primary != INCUMBENT else "H1b_v1 retained")
log.append(f"Primary vs H1b_v1 (held-out MAE): {M.loc[primary, 'MAE'] - M.loc[INCUMBENT, 'MAE']:+.2f} "
           f"[{lo:+.2f}, {hi:+.2f}] -> {claim}")

full_leader = rep_full.set_index("system").loc[ELIGIBLE, "MAE"].idxmin()
log.append(f"Full-val MAE leader (comparison): {full_leader}"
           + ("  (same as held-out leader)" if full_leader == leader else "  (DIFFERS from held-out leader)"))

print("=" * 80); print("B4c SELECTION (pre-declared rule, held-out)"); print("=" * 80)
print("\n".join(log))
print("\nEffect sizes vs PRIMARY (held-out MAE, A − primary, 95% CI):")
for c in SYSTEMS_ALL:
    if c != primary:
        a, b = ci(BOOT_HO, c, primary, "MAE")
        print(f"  {c:<16}: {M.loc[c, 'MAE'] - M.loc[primary, 'MAE']:+.2f}  [{a:+.2f}, {b:+.2f}]")

B4c SELECTION (pre-declared rule, held-out)
Stage 1 leader (lowest MAE): GPV_R4+P[MixA] (27.07)
Stage 1 tied set (MAE CI vs leader includes 0): ['GPV_R4+P[MixA]', 'GPV_K-n+P[LM0]']
Stage 1 after RMSE filter (best RMSE GPV_K-n+P[LM0]): ['GPV_R4+P[MixA]', 'GPV_K-n+P[LM0]']
Stage 2 failures: GPV_R4+P[MixA]: none; GPV_K-n+P[LM0]: ['trimmed MAE worse than GPV_R4+P[MixA]', "worse than H1b_v1 in ['S4', 'S5']"]
Stage 2 remaining: ['GPV_R4+P[MixA]']
Stage 3 order ['GPV_K-n+P[LM0]', 'GPV_K-n+P[MixA]', 'GPV_R4+P[MixA]', 'H1b+P[MixA]', 'H1b-K3n+P[LM0]', 'H1b_v1']
PRIMARY = GPV_R4+P[MixA]; FALLBACK = H1b+P[MixA] (only one survivor: lowest-MAE other eligible)
Clear winner (MAE better than every other eligible, CI excludes 0): none
Primary vs H1b_v1 (held-out MAE): -2.92 [-3.98, -1.91] -> significantly better than H1b_v1
Full-val MAE leader (comparison): H1b+P[MixA]  (DIFFERS from held-out leader)

Effect sizes vs PRIMARY (held-out MAE, A − primary, 95% CI):
  B0              : +11.28  [+9.54, +13.15

In [10]:
# ============================================================
# Step C.4 — Secondary analysis: geometric-prior contribution
#            GPV_K-n+P[MixA] − GPV_R4+P[MixA] (same mask set, normals K3n on/off)
# ============================================================
GEO_A, GEO_B = "GPV_K-n+P[MixA]", "GPV_R4+P[MixA]"
geo_rows = []
for label, B, rep in [("held-out", BOOT_HO, rep_ho), ("full val", BOOT_FULL, rep_full)]:
    R = rep.set_index("system")
    for m in ["MAE", "RMSE", "trimmed_MAE"] + SCOLS:
        lo, hi = ci(B, GEO_A, GEO_B, m)
        geo_rows.append({"set": label, "metric": m, "K-n − R4": R.loc[GEO_A, m] - R.loc[GEO_B, m],
                         "CI_low": lo, "CI_high": hi, "significant": not (lo <= 0 <= hi)})
geo_df = pd.DataFrame(geo_rows)
display(geo_df.round(2))
g = (ho[GEO_A] != ho[GEO_B])
print(f"Held-out images where normals change the count: {int(g.sum())} / {len(ho)}")

,set,metric,K-n − R4,CI_low,CI_high,significant
0,held-out,MAE,1.98,1.03,2.95,True
1,held-out,RMSE,-0.93,-5.82,2.27,False
2,held-out,trimmed_MAE,2.23,1.42,3.13,True
3,held-out,MAE_S1,6.49,3.69,9.69,True
4,held-out,MAE_S2,3.15,1.18,5.69,True
5,held-out,MAE_S3,0.62,-0.09,1.52,False
6,held-out,MAE_S4,2.25,1.08,3.67,True
7,held-out,MAE_S5,-1.35,-2.43,-0.48,True
8,held-out,MAE_S6,-6.91,-17.31,-1.24,True
9,full val,MAE,2.38,1.22,3.73,True


Held-out images where normals change the count: 141 / 1186


In [11]:
# ============================================================
# Step C.5 — Save
# ============================================================
ho.to_csv(OUT_DIR / "B4c_heldout_per_image.csv", index=False)
full.to_csv(OUT_DIR / "B4c_fullval_per_image.csv", index=False)
pd.concat([rep_ho, rep_full, rep_dev]).to_csv(OUT_DIR / "B4c_metrics.csv", index=False)
pairs_ho.to_csv(OUT_DIR / "B4c_bootstrap_pairs_heldout.csv", index=False)
vs_h1b.to_csv(OUT_DIR / "B4c_heldout_vs_H1b_v1.csv"); vs_b0.to_csv(OUT_DIR / "B4c_heldout_vs_B0.csv")
gaps.to_csv(OUT_DIR / "B4c_heldout_stratum_gaps.csv")
geo_df.to_csv(OUT_DIR / "B4c_secondary_geometry.csv", index=False)
with open(OUT_DIR / "B4c_selection.json", "w") as f:
    json.dump({"date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "module": C.VERSION, "rule_file": RULE_PATH.name,
               "selection_log": log, "primary": primary, "fallback": fallback,
               "clear_winner": clear, "primary_vs_H1b_v1": claim, "full_val_leader": full_leader,
               "n_boot": N_BOOT, "seed": 0, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)
print(f"\nPRIMARY FOR B5: {primary}   (fallback: {fallback})")

Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B4c_HeldOut_Candidates

PRIMARY FOR B5: GPV_R4+P[MixA]   (fallback: H1b+P[MixA])


In [7]:
# ============================================================
# Step D.1 — Diagnostic (report only): empty candidate mask sets
# ============================================================
emp = ho[(ho["n_r4[MixA]"] == 0) | (ho["n_r4[LM0]"] == 0)].copy()
def region_frac(r):
    with np.load(HO_ROOT / "san_vocab_tiled" / f"{io_cache.image_stem(r.image_id)}.npz") as v:
        return float((v["tiled_argmax"] == 0).mean())
emp["region_frac"] = [region_frac(r) for r in emp.itertuples()]
display(emp[["dataset_index", "image_id", "category", "gt_count", "density_stratum",
             "region_frac", "n_r4[MixA]", "n_r4[LM0]", "B0", "H1b_v1"]].sort_values("gt_count"))
print("By stratum:", emp["density_stratum"].value_counts().reindex(STRATA, fill_value=0).to_dict())
print(f"Contribution to candidate held-out MAE: {emp['gt_count'].sum() / len(ho):.2f}")
others = ho.drop(emp.index)
print(f"GPV_R4+P[MixA] MAE excluding them: {np.abs(others['GPV_R4+P[MixA]'] - others['gt_count']).mean():.2f}"
      f"  | H1b_v1: {np.abs(others['H1b_v1'] - others['gt_count']).mean():.2f}")
tr = pd.DataFrame([json.load(open(p)) for p in sorted((OUT_DIR / "A_train_rows").glob("*.json"))])
print("Train pool empty sets:", {b: int((tr[f"n_r4[{b}]"] == 0).sum()) for b in C.BASES})
emp.to_csv(OUT_DIR / "B4c_D_empty_mask_sets.csv", index=False)

,dataset_index,image_id,category,gt_count,density_stratum,region_frac,n_r4[MixA],n_r4[LM0],B0,H1b_v1
998,1083,7532.jpg,fresh cut,20,S2_20_40,0.000000,0,0,13,20.0
731,798,4855.jpg,grapes,22,S3_40_60,0.000000,0,0,29,31.0
999,1084,6927.jpg,fresh cut,32,S3_40_60,0.000000,0,0,48,46.0
791,863,5151.jpg,toilet paper rolls,32,S3_40_60,0.000000,0,0,8,13.0
1002,1087,7330.jpg,fresh cut,40,S4_60_80,0.000000,0,0,41,46.0
845,922,5742.jpg,polka dots,43,S4_60_80,0.000000,0,0,5,8.0
1007,1092,7334.jpg,fresh cut,53,S4_60_80,0.000000,0,0,50,68.0
1001,1086,7228.jpg,fresh cut,63,S4_60_80,0.000000,0,0,77,79.0
1004,1089,7540.jpg,fresh cut,64,S4_60_80,0.000000,0,0,45,49.0
1006,1091,7428.jpg,fresh cut,66,S4_60_80,0.000000,0,0,75,75.0


By stratum: {'S1_0_20': 0, 'S2_20_40': 1, 'S3_40_60': 3, 'S4_60_80': 7, 'S5_80_95': 8, 'S6_95_100': 1}
Contribution to candidate held-out MAE: 1.64
GPV_R4+P[MixA] MAE excluding them: 25.87  | H1b_v1: 29.55
Train pool empty sets: {'MixA': 5, 'LM0': 5}


In [8]:
# ============================================================
# Step D.2 — Diagnostic (report only): which vocabulary term wins on empty-region images
# ============================================================
import src.pipeline.semantic as semantic
ref4 = pd.read_csv(B4_C / "B4_C_heldout_per_image.csv")[["dataset_index", "GPV_R4_v1"]]
rows = []
for r in emp.merge(ref4, on="dataset_index").itertuples():
    vocab = semantic.expanded_vocabulary(r.category)
    with np.load(HO_ROOT / "san_vocab_tiled" / f"{io_cache.image_stem(r.image_id)}.npz") as v:
        amax = v["tiled_argmax"]
    cls, cnt = np.unique(amax, return_counts=True)
    top = sorted(zip(cnt, cls), reverse=True)[:3]
    rows.append({"image_id": r.image_id, "category": r.category, "gt": r.gt_count,
                 "top_terms": ", ".join(f"{vocab[c]} {n / amax.size:.0%}" for n, c in top),
                 "B0": r.B0, "H1b_v1": r.H1b_v1, "GPV_R4_v1": r.GPV_R4_v1})
d2 = pd.DataFrame(rows).sort_values(["category", "gt"])
display(d2)
for s in ["B0", "H1b_v1", "GPV_R4_v1"]:
    print(f"{s:<10} error on these images: {np.abs(d2[s] - d2['gt']).sum() / len(ho):.2f} MAE-equivalent "
          f"(candidates: {d2['gt'].sum() / len(ho):.2f})")
d2.to_csv(OUT_DIR / "B4c_D2_empty_region_winners.csv", index=False)

,image_id,category,gt,top_terms,B0,H1b_v1,GPV_R4_v1
0,2858.jpg,chairs,131,"roof 65%, sea 32%, sky 3%",2,59.0,38
9,7532.jpg,fresh cut,20,"counter 92%, background 8%",13,20.0,20
10,6927.jpg,fresh cut,32,"counter 71%, shelf 29%, background 0%",48,46.0,51
13,7330.jpg,fresh cut,40,counter 100%,41,46.0,45
18,7334.jpg,fresh cut,53,"counter 80%, shelf 20%, background 0%",50,68.0,67
12,7228.jpg,fresh cut,63,"shelf 96%, counter 4%",77,79.0,81
15,7540.jpg,fresh cut,64,"counter 98%, shelf 1%, wall 0%",45,49.0,45
17,7428.jpg,fresh cut,66,"shelf 64%, cabinet 32%, counter 4%",75,75.0,75
11,6728.jpg,fresh cut,99,"shelf 89%, counter 11%, background 0%",69,89.0,82
14,6779.jpg,fresh cut,119,"counter 75%, shelf 25%",95,99.0,98


B0         error on these images: 1.12 MAE-equivalent (candidates: 1.64)
H1b_v1     error on these images: 0.94 MAE-equivalent (candidates: 1.64)
GPV_R4_v1  error on these images: 1.02 MAE-equivalent (candidates: 1.64)


In [9]:
# ============================================================
# Step D.3 — Feasibility (report only): does the [class, background] SAN region exist
#            where the expanded-vocabulary region is empty? (train pool + held-out)
# ============================================================
def region2_frac(san_dir, image_id):
    s = io_cache.load_san_scores(Path(san_dir) / io_cache.san_scores_filename(image_id))
    assert s["vocabulary"][:2][1] == "background", s["vocabulary"]
    return float((s["target_score"] > s["background_score"]).mean())

tr_emp = tr[tr["n_r4[MixA]"] == 0].merge(td_df[["dataset_index", "image_id", "category"]], on="dataset_index")
d3 = pd.concat([
    pd.DataFrame({"pool": "train", "image_id": tr_emp["image_id"], "category": tr_emp["category"],
                  "region2_frac": [region2_frac(TD_ROOT / "san", i) for i in tr_emp["image_id"]]}),
    pd.DataFrame({"pool": "held-out", "image_id": emp["image_id"], "category": emp["category"],
                  "region2_frac": [region2_frac(HO_ROOT / "san", i) for i in emp["image_id"]]}),
], ignore_index=True)
display(d3.sort_values(["pool", "category"]))
print("Non-empty [class, background] region:",
      d3.groupby("pool")["region2_frac"].apply(lambda x: f"{(x > 0).sum()}/{len(x)}").to_dict())
d3.to_csv(OUT_DIR / "B4c_D3_region2_feasibility.csv", index=False)

,pool,image_id,category,region2_frac
5,held-out,2858.jpg,chairs,0.979827
14,held-out,7532.jpg,fresh cut,0.000000
15,held-out,6927.jpg,fresh cut,0.000000
16,held-out,6728.jpg,fresh cut,0.000000
17,held-out,7228.jpg,fresh cut,0.000000
18,held-out,7330.jpg,fresh cut,0.000000
19,held-out,6779.jpg,fresh cut,0.000000
20,held-out,7540.jpg,fresh cut,0.000000
21,held-out,6885.jpg,fresh cut,0.000000
22,held-out,7428.jpg,fresh cut,0.000000


Non-empty [class, background] region: {'held-out': '4/20', 'train': '4/5'}


## B4c — Conclusion (14–15 Oct 2026)

**Purpose.** Second, pre-declared use of the FSC-147 val held-out set (the first was B4): compare the five improvement-phase candidates (developed on train_dev400, B3S–B3AC) with H1b_v1 (B4 choice) and B0 (reference), and select the architecture(s) for B5/B6.

### Part A — Frozen module
`src/pipeline/candidates.py` (`candidates_v1`) reproduced the train_dev400 per-image counts of B3PN and B3AC **exactly** (9/9 comparisons, 400/400 images). Frozen configs written to `configs/`.

### Pre-declared rule
Recorded in `B4c_predeclared_rule.json` (14 Oct 2026, 08:53 UTC) **before** any validation count was computed. Decision set = held-out (1,186); full val (1,286) for comparison only; test untouched.
- Stage 1 (performance): MAE leader, CI tie set, RMSE filter.
- Stage 2 (stability): trimmed MAE; significantly worse than H1b_v1 in ≥ 2 of 6 strata; overall MAE significantly worse than H1b_v1.
- Stage 3 (multi-label adaptability, both priors, cost): GPV_K-n+P[LM0] → GPV_K-n+P[MixA] → GPV_R4+P[MixA] → H1b+P[MixA] → H1b-K3n+P[LM0] → H1b_v1.

### Held-out results (1,186)

| System | MAE | RMSE | Trim. MAE | NAE | S1 | S2 | S3 | S4 | S5 | S6 |
|---|---|---|---|---|---|---|---|---|---|---|
| B0 (reference) | 38.35 | 102.23 | 31.04 | 0.53 | 6.03 | 8.02 | 11.78 | 25.95 | 74.67 | 356.58 |
| **GPV_R4+P[MixA]** | **27.07** | 88.33 | **20.23** | 0.42 | 6.74 | 7.00 | 8.86 | 14.76 | 41.10 | 286.53 |
| GPV_K-n+P[MixA] | 29.05 | 87.40 | 22.46 | 0.59 | 13.23 | 10.15 | 9.49 | 17.01 | 39.76 | 279.62 |
| GPV_K-n+P[LM0] | 28.43 | **85.73** | 22.35 | 0.38 | 4.50 | 7.13 | 7.71 | 16.24 | 48.53 | 299.07 |
| H1b+P[MixA] | 27.56 | 86.87 | 20.85 | 0.48 | 9.05 | 7.46 | 9.12 | 16.51 | 40.85 | 277.38 |
| H1b-K3n+P[LM0] | 29.14 | 94.42 | 21.99 | 0.35 | 4.03 | 5.56 | 7.61 | 16.07 | 49.55 | 320.89 |
| H1b_v1 (incumbent) | 29.99 | 90.60 | 23.07 | 0.52 | 9.58 | 8.82 | 10.33 | 14.61 | 44.90 | 311.18 |

### Selection (pre-declared rule)
- **Stage 1:** leader GPV_R4+P[MixA] (27.07); tied: GPV_K-n+P[LM0] only (difference −1.36, CI [−2.93, +0.55]). H1b+P[MixA] excluded: +0.49, CI [+0.08, +0.91]. RMSE filter removed none.
- **Stage 2:** GPV_K-n+P[LM0] removed (trimmed MAE worse; significantly worse than H1b_v1 in S4 and S5). GPV_R4+P[MixA]: no failures.
- **Outcome:** **PRIMARY = GPV_R4+P[MixA]**; rule fallback = H1b+P[MixA] (only one survivor → lowest-MAE other eligible). Stage 3 was not needed.
- **Primary vs H1b_v1:** −2.92 MAE [−3.98, −1.91], significant (≈ 10%); vs B0: −11.28 [−13.15, −9.54] (≈ 29%). Not significantly worse than H1b_v1 in any stratum; significantly better in S1, S2, S3, S5, S6. Matches B0 on S1 (+0.72, n.s.).

### Post-hoc decision: secondary (geometric) system
Both **GPV_R4+P[MixA] (primary)** and **GPV_K-n+P[LM0] (secondary)** go to B5 and B6. This **deviates from the rule's fallback** (H1b+P[MixA]) and is a post-hoc decision, recorded here before B5. Reasons:
1. GPV_K-n+P[LM0] is the only system statistically tied with the primary on MAE (H1b+P[MixA] is significantly worse).
2. It keeps a geometric prior (Marigold normals, K3n) in the evaluation; without it B5/B6 would carry no geometry.
3. Complementary error profile: best sparse performance among GPV systems (S1 4.50, significantly better than B0); the primary is stronger in S5–S6.
4. GPV family: no A3c/SG gate, normals class-agnostic → far easier to adapt to multi-label than the H1b family.

Caveats: GPV_K-n+P[LM0] failed Stage 2 (S4, S5 vs H1b_v1; trimmed MAE; wide S6 CI). The headline result in B5/B6 is the **primary**, fixed before test. R4 vs K-n[LM0] differs in both mask set and K3n, so their gap does **not** isolate the geometric prior.

### Secondary analysis — geometric-prior contribution
GPV_K-n+P[MixA] − GPV_R4+P[MixA] (same masks; normals K3n on/off; count changes on 141/1,186 images): helps dense scenes (S5 −1.35, S6 −6.91, both significant) but hurts S1 (+6.49), S2 (+3.15), S4 (+2.25); net **+1.98 MAE, significant**. The geometric cue carries real dense-scene signal, but the multiplicity gate is not selective enough to give a net gain.

### Other observations
- **Winner's curse confirmed.** Train-pool order K-n[MixA] < H1b+P < R4 < K-n[LM0] < H1b-K3n became R4 < H1b+P < K-n[LM0] < K-n[MixA] < H1b-K3n on held-out; gated systems degraded most (K-n[MixA] S1: 10.68 → 13.23). This justifies the second held-out comparison.
- **Dense undercounting remains the dominant error.** All systems have negative mean signed error (primary −15.1); S5+S6 (20% of images) give ≈ 72% of the primary's MAE (S6 alone ≈ 13.3 of 27.07).
- **Full val** leader is H1b+P[MixA] (29.79 vs 30.01) because dev100 over-samples dense images (10% S6 vs 4.6%). Full val is not population-weighted; held-out decides.

### D.1–D.3 — Empty semantic region (report only, no re-run)
20 held-out images (1.7%; train pool 5/400) have an empty candidate mask set: the tiled expanded-vocabulary region is empty (`region_frac` = 0), every SAM2 mask is discarded, and all candidates count 0. Cost: 1.64 MAE for every candidate. Excluding these images (post-hoc, reporting only) the primary scores 25.87 vs H1b_v1 29.55, so the reported improvement is conservative. The selection is unaffected (all candidates share the region).

**Mechanism (D.2: winning vocabulary terms; D.3: two-class [class, background] region).** "Background" almost never wins; scene/surface stuff terms do. Three failure types:
- **Scene absorption** (chairs → roof/sea; grapes → table; toilet paper rolls → wall/cupboard): the two-class region recovers all 3.
- **Surface / texture classes** (polka dots → wall/floor/dirt; train pool: bricks, windows, mosaic tiles): the two-class region recovers train 4/5, polka dots 1/6.
- **Concept failure** (fresh cut, 11 images → counter/shelf): the target loses even to "background" alone (two-class region empty on 11/11); unmasked systems count these well (e.g. GPV_R4_v1 20/20, 45/40, 75/66).

On these 20 images, unmasked systems err less (H1b_v1 0.94, GPV_R4_v1 1.02, B0 1.12 MAE-equivalent vs 1.64): a fallback could recover ≈ 0.6 MAE (illustrative, post-hoc).

**Decision.** The hard semantic gate needs graceful degradation. This is addressed together with multi-label adaptation in B5 as one per-class region-resolution component with a GT-free, parameter-free fallback hierarchy: R0 joint vocabulary (= current region) → R1 without stuff terms (single-class: two-class SAN region) → R2 residual (single-class: whole image). Each level fires only if the previous level's mask set is empty. Regression checks: with fallback off, single-class FSC must reproduce v1 exactly; with fallback on, FSC v1.1 results are reported as descriptive/post-hoc. **The B4c selection (v1) stands.** Known limitation: a small non-empty region can still undercount without triggering a fallback.

### Next
1. Commit this notebook.
2. B5.0: OmniCount-191 val clean subset (excluding the 62 test-overlapping images); joint-vocabulary SAN; per-(image, class) empty-region rate at R0/R1/R2.
3. B5.1: `candidates_v2` (per-class regions + fallback hierarchy) for GPV_R4+P[MixA] (primary) and GPV_K-n+P[LM0] (secondary); FSC single-class regression (fallback off = v1 exactly; fallback on = v1.1 descriptive).
4. B5.2–B5.3: multi-label caching and counting on OmniCount val; freeze module and configs.
5. B6: FSC-147 test and OmniCount-191 test with roles fixed (primary = headline); competitor comparison.